# D-GRAG Demo Notebook

This notebook provides a **mocked, end-to-end walkthrough** of the Delta-Graph Retrieval-Augmented Generation (D-GRAG) workflow using only local deterministic components.

It is designed to:

- avoid any real external API dependency,
- run on a tiny synthetic repository,
- build a call graph,
- parse a PR-style diff,
- run the review pipeline with a mock LLM response,
- visualize the impact subgraph,
- and inspect the final review output as a table.

> Recommended environment for this demo: `MOCK_LLM=true`


## Notebook outline

1. Verify imports and local environment
2. Create a tiny demo repository in a temporary directory
3. Build a static call graph for that repository
4. Prepare a PR-style unified diff
5. Run the D-GRAG review pipeline with a deterministic mock review payload
6. Visualize the resulting impact subgraph
7. Display findings and summary data as tables


In [ ]:
from __future__ import annotations

import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from src.call_graph_builder import build_call_graph
from src.pipeline.review_pipeline import (
    PipelineConfig,
    run_review_pipeline,
    summarize_pipeline_result,
)

print("Python executable available")
print(f"MOCK_LLM={os.getenv('MOCK_LLM', 'true')}")
print("Notebook imports loaded successfully.")


## Create a tiny demo repository

We create a very small Python repo with a few functions so that the graph and impact flow are easy to understand.


In [ ]:
demo_root = Path(tempfile.mkdtemp(prefix="dgrag_demo_"))

def write(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

write(
    demo_root / "helpers.py",
    "def helper(value):\n"
    "    return value + 1\n"
    "\n"
    "def validate(value):\n"
    "    if value < 0:\n"
    "        raise ValueError('negative values are not allowed')\n"
    "    return value\n",
)

write(
    demo_root / "service.py",
    "from helpers import helper, validate\n"
    "\n"
    "def compute(value):\n"
    "    checked = validate(value)\n"
    "    return helper(checked)\n",
)

write(
    demo_root / "app.py",
    "from service import compute\n"
    "\n"
    "def run(value):\n"
    "    return compute(value)\n"
)

print(f"Demo repository created at: {demo_root}")
for path in sorted(demo_root.rglob('*.py')):
    print('-', path.relative_to(demo_root))


## Build the static call graph

Now we build the repository call graph used by D-GRAG retrieval.


In [ ]:
call_graph = build_call_graph(demo_root)

print(f"Graph nodes: {call_graph.number_of_nodes()}")
print(f"Graph edges: {call_graph.number_of_edges()}")

nodes_df = pd.DataFrame(
    [
        {
            "node_id": node_id,
            "qualified_name": attrs.get("qualified_name"),
            "file": attrs.get("file_path") or attrs.get("file"),
            "start_line": attrs.get("start_line"),
            "end_line": attrs.get("end_line"),
        }
        for node_id, attrs in call_graph.nodes(data=True)
    ]
).sort_values(by=["file", "start_line", "qualified_name"], kind="stable")

nodes_df


## Prepare a PR-style unified diff

The synthetic PR below changes the `helper` logic. D-GRAG should map the modified lines back into the graph and pull in relevant callers/callees.


In [ ]:
pr_diff = """\
diff --git a/helpers.py b/helpers.py
index 1111111..2222222 100644
--- a/helpers.py
+++ b/helpers.py
@@ -1,2 +1,2 @@
-def helper(value):
-    return value + 1
+def helper(value):
+    return value + 2
"""

print(pr_diff)


## Run the D-GRAG review pipeline with a mocked LLM response

We configure the full review path, but use a deterministic mock JSON response so the notebook can run without any real model or API key.


In [ ]:
mock_review_payload = {
    "overall_risk": "medium",
    "findings": [
        {
            "category": "correctness",
            "severity": "medium",
            "confidence": 0.92,
            "summary": "Behavior change in helper may affect downstream callers",
            "technical_reasoning": "The helper return value changed from +1 to +2, which can alter expectations in compute() and run().",
            "suggested_fix": "Confirm the new return value is intentional and update dependent assumptions or tests.",
            "evidence": [
                {
                    "node_id": "helper",
                    "file_path": "helpers.py",
                    "start_line": 1,
                    "end_line": 2,
                }
            ],
        }
    ],
}

config = PipelineConfig(
    k_up=2,
    k_down=2,
    max_nodes=25,
    max_edges=50,
    max_per_anchor=20,
    max_chars=12000,
    include_code=True,
    include_diff_in_context=True,
    run_full_review=True,
    strict_json_output=True,
    llm_backend="mock",
    llm_model_name="demo-mock-reviewer",
    llm_temperature=0.0,
    llm_max_new_tokens=512,
    llm_mock_response_text=json.dumps(mock_review_payload),
    allow_dev_mock_controls=True,
    output_format="json",
)

result = run_review_pipeline(
    call_graph=call_graph,
    pr_diff=pr_diff,
    config=config,
    pr_metadata={
        "pr_id": "demo-1",
        "title": "Change helper arithmetic",
        "description": "Synthetic PR used for notebook demonstration.",
    },
)

summary = summarize_pipeline_result(result)
print("Pipeline finished.")
print(json.dumps(summary, indent=2))


## Inspect the linearized context

This is the structured context D-GRAG assembled around the changed function.


In [ ]:
print(result.linearized_context)


## Visualize the impact subgraph

We draw the retrieved impact subgraph using NetworkX and Matplotlib.


In [ ]:
impact_graph = result.impact_subgraph

plt.figure(figsize=(8, 5))
pos = nx.spring_layout(impact_graph, seed=7)

labels = {
    node: impact_graph.nodes[node].get("qualified_name", node)
    for node in impact_graph.nodes()
}

nx.draw_networkx_nodes(impact_graph, pos, node_color="#4C78A8", node_size=1600)
nx.draw_networkx_edges(impact_graph, pos, arrows=True, arrowstyle="-|>", width=1.5)
nx.draw_networkx_labels(impact_graph, pos, labels=labels, font_size=9)

plt.title("D-GRAG Impact Subgraph")
plt.axis("off")
plt.show()


## Display structured review output

The normalized review payload can be inspected as a DataFrame for notebook-friendly analysis.


In [ ]:
review_df = pd.DataFrame(result.normalized_review["findings"])
review_df


## Summary metrics

A compact run summary is often helpful for debugging and evaluation.


In [ ]:
summary_df = pd.DataFrame([summary])
summary_df


## Final formatted review artifact

Because this notebook uses the mocked review path, the output below is deterministic and safe for demos.


In [ ]:
print(result.formatted_review)


## Notes

- This notebook intentionally avoids external network calls.
- It uses the mock LLM path through the pipeline configuration.
- It demonstrates the architecture, not a benchmark claim.
- To turn this into a live GitHub-backed or model-backed workflow, use the CLI/webhook paths outside the notebook with real environment variables.
